In [1]:
import numpy as np
import pandas as pd

In [60]:
df = pd.read_csv('./titanic_train.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [61]:
df = df.dropna(subset=["Embarked", "Age", "Survived"])

In [5]:
ser = df['Embarked']

def calc_gini(ser):
    ser_len = ser.count()
    probabilies = ser.value_counts() / ser_len
    inv_probabilities = 1 - probabilies    
    return np.dot(inv_probabilities, probabilies)

def calc_gini2(ser):
    ser_len = ser.count()
    probabilies = ser.value_counts() / ser_len
    return 1 - np.sum(probabilies ** 2)    
    
(calc_gini(ser), calc_gini2(ser))

(np.float64(0.3596925893195304), np.float64(0.3596925893195305))

In [6]:
ser = df['Embarked']

def calc_entropy(ser):
    ser_len = ser.count()
    probabilies = ser.value_counts() / ser_len
    
    return np.abs(np.sum(np.log2(probabilies) * probabilies))

calc_entropy(ser)

np.float64(0.9131956311032485)

In [7]:
X, y = df[['Fare']], df['Survived']

In [8]:
#X['Embarked'] = df['Embarked'].apply(lambda x: 0 if x == 'S' else 1 if x == 'C' else 2)
X['Sex'] = df['Sex'].apply(lambda x: 0 if x == 'male' else 1)

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

clf = DecisionTreeClassifier(random_state=15)
clf.fit(X, y)

tree_rules = export_text(
    clf, 
    feature_names=['Sex'],       # Передаем реальные названия колонок
    spacing=4,                    # Количество пробелов для отступов (удобно для чтения)
    decimals=2                    # Округление пороговых значений split
)

# 3. Выводим результат
print(tree_rules)

In [28]:
curr_feat = df['Fare']
y = df["Survived"]
apply_info_criterion = calc_gini

# unique_thresholds = np.sort(np.unique(curr_feat)) np.unique returns sorted values. np.sort is useless in this case
unique_thresholds = np.unique(curr_feat)
feat_elemenets_count = curr_feat.count()
curr_info_param = apply_info_criterion(y)

best_info_gain = 0
best_threshold = None

if len(unique_thresholds) == 0:
    pass

for threshold_index in range(len(unique_thresholds) - 1):
    threshold = (unique_thresholds[threshold_index + 1] - unique_thresholds[threshold_index]) / 2 + unique_thresholds[threshold_index]
    left_mask = curr_feat <= threshold
    right_mask = curr_feat > threshold

    left_group = y[left_mask]
    right_group = y[right_mask]

    left_info_param = apply_info_criterion(left_group)
    right_info_param = apply_info_criterion(right_group)
    probability_to_be_put_to_left_group = left_group.count() / feat_elemenets_count
    probability_to_be_put_to_right_group = right_group.count() / feat_elemenets_count

    weighted_info_param = probability_to_be_put_to_left_group * left_info_param \
        + probability_to_be_put_to_right_group * right_info_param

    info_gain = curr_info_param - weighted_info_param

    if info_gain > best_info_gain:
        best_info_gain = info_gain
        best_threshold = threshold
(best_info_gain, best_threshold)

(np.float64(0.045469752380312145), np.float64(52.277100000000004))

In [80]:
X, y = df[['Fare', 'Sex']], df['Survived']
X['Sex'] = X['Sex'].apply(lambda x: 0 if x == 'male' else 1)
from sklearn.tree import DecisionTreeClassifier, export_text

clf = DecisionTreeClassifier(random_state=15)
clf.fit(X, y)

tree_rules = export_text(
    clf, 
    feature_names=['Fare', 'Sex'],       # Передаем реальные названия колонок
    spacing=4,                    # Количество пробелов для отступов (удобно для чтения)
    decimals=2                    # Округление пороговых значений split
)

# 3. Выводим результат
print(tree_rules)

|---- Sex <= 0.50
|    |---- Fare <= 26.27
|    |    |---- Fare <= 7.91
|    |    |    |---- Fare <= 7.86
|    |    |    |    |---- Fare <= 7.79
|    |    |    |    |    |---- Fare <= 7.24
|    |    |    |    |    |    |---- Fare <= 7.13
|    |    |    |    |    |    |    |---- Fare <= 7.01
|    |    |    |    |    |    |    |    |---- Fare <= 6.86
|    |    |    |    |    |    |    |    |    |---- Fare <= 2.01
|    |    |    |    |    |    |    |    |    |    |---- class: 0
|    |    |    |    |    |    |    |    |    |---- Fare >  2.01
|    |    |    |    |    |    |    |    |    |    |---- class: 0
|    |    |    |    |    |    |    |    |---- Fare >  6.86
|    |    |    |    |    |    |    |    |    |---- class: 0
|    |    |    |    |    |    |    |---- Fare >  7.01
|    |    |    |    |    |    |    |    |---- class: 0
|    |    |    |    |    |    |---- Fare >  7.13
|    |    |    |    |    |    |    |---- Fare <= 7.18
|    |    |    |    |    |    |    |    |---- class: 1
|    

In [125]:
feats = df[['Fare', 'Sex']]
feats['Sex'] = feats['Sex'].apply(lambda x: 0 if x == 'male' else 1)
y = df["Survived"]


def best_split(feats, y):
    apply_info_criterion = calc_gini
    
    best_info_gain = 0
    best_threshold = None
    best_feat = None
    
    for col in feats.columns:
        curr_feat = feats[col]
        unique_thresholds = np.unique(curr_feat)
        feat_elemenets_count = curr_feat.count()
        curr_info_param = apply_info_criterion(y)
        
        if len(unique_thresholds) == 0:
            continue
        
        for threshold_index in range(len(unique_thresholds) - 1):
            threshold = (unique_thresholds[threshold_index + 1] - unique_thresholds[threshold_index]) / 2 + unique_thresholds[threshold_index]
            left_mask = curr_feat <= threshold
            right_mask = curr_feat > threshold
        
            left_group = y[left_mask]
            right_group = y[right_mask]
        
            left_info_param = apply_info_criterion(left_group)
            right_info_param = apply_info_criterion(right_group)
            probability_to_be_put_to_left_group = left_group.count() / feat_elemenets_count
            probability_to_be_put_to_right_group = right_group.count() / feat_elemenets_count
        
            weighted_info_param = probability_to_be_put_to_left_group * left_info_param \
                + probability_to_be_put_to_right_group * right_info_param
        
            info_gain = curr_info_param - weighted_info_param
        
            if info_gain > best_info_gain:
                best_info_gain = info_gain
                best_threshold = threshold
                best_feat = col
                
    return best_info_gain, best_threshold, best_feat
best_split(feats, y)

(np.float64(0.13880057591007855), np.float64(0.5), 'Sex')

In [84]:
# write the tree
from collections import Counter

def get_most_common_el(vect):
    return Counter(vect).most_common(1)[0][0]

class Node:
    def __init__(
        self, 
        value = None, 
        left = None,
        right = None,
        threshold = None,
        feature = None
    ):
        self.value = value
        self.left = left
        self.right = right
        self.threshold = threshold
        self.feature = feature 
        
class MyDecisionTreeClassifier:
    def __init__(self, criterion='gini'):
        self.criterion = criterion
        self.root = None

    def _split(self, X, y):
        return best_split(X, y)
        
    def fit(self, X, y):
        self.root = self._build_tree(X, y)
        return self

    def _build_tree(self, X, y):
        best_info_gain, best_threshold, best_feat = self._split(X, y)

        if best_feat is None:
            return Node(value=get_most_common_el(y))

        left_mask = X[best_feat] <= best_threshold
        right_mask = X[best_feat] > best_threshold

        left_child = self._build_tree(X[left_mask], y[left_mask])
        right_child = self._build_tree(X[right_mask], y[right_mask])

        return Node(left=left_child, right=right_child, feature=best_feat, threshold=best_threshold)

feats = df[['Fare', 'Sex']]
feats['Sex'] = feats['Sex'].apply(lambda x: 0 if x == 'male' else 1)
y = df["Survived"]
clf = MyDecisionTreeClassifier()
clf.fit(feats, y)


In [121]:
X, y = df[['Fare', 'Sex']], df['Survived']
X['Sex'] = X['Sex'].apply(lambda x: 0 if x == 'male' else 1)
from sklearn.tree import DecisionTreeClassifier, export_text

clf = DecisionTreeClassifier(random_state=15, max_depth=3)
clf.fit(X, y)

tree_rules = export_text(
    clf, 
    feature_names=['Fare', 'Sex'],       # Передаем реальные названия колонок
    spacing=4,                    # Количество пробелов для отступов (удобно для чтения)
    decimals=2                    # Округление пороговых значений split
)

# 3. Выводим результат
print(tree_rules)
clf.predict(X)

|---- Sex <= 0.50
|    |---- Fare <= 26.27
|    |    |---- Fare <= 7.91
|    |    |    |---- class: 0
|    |    |---- Fare >  7.91
|    |    |    |---- class: 0
|    |---- Fare >  26.27
|    |    |---- Fare <= 26.47
|    |    |    |---- class: 1
|    |    |---- Fare >  26.47
|    |    |    |---- class: 0
|---- Sex >  0.50
|    |---- Fare <= 48.20
|    |    |---- Fare <= 10.48
|    |    |    |---- class: 1
|    |    |---- Fare >  10.48
|    |    |    |---- class: 1
|    |---- Fare >  48.20
|    |    |---- Fare <= 149.04
|    |    |    |---- class: 1
|    |    |---- Fare >  149.04
|    |    |    |---- class: 1



array([0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1,
       1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1,
       0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0,
       1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1,
       1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0,
       0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0,
       0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0,
       1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1,
       1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0,
       1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0,
       0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1,
       1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1,
       1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0,

In [97]:
# with max_depth and different criterion
class MyDecisionTreeClassifier:
    def __init__(self, criterion='gini', max_depth=None):
        if max_depth is not None and not isinstance(max_depth, int):
            raise TypeError('max_depth must be int')
        if max_depth is not None and max_depth < 1:
            raise ValueError('max_depth must be more or equal to one (1)')
            
        self.criterion = criterion
        self.root = None
        self.max_depth = max_depth

    def _split(self, X, y):
        return best_split(X, y)
        
    def fit(self, X, y):
        self.root = self._build_tree(X, y, 0)
        return self

    def _build_tree(self, X, y, depth):
        best_info_gain, best_threshold, best_feat = self._split(X, y)

        if best_feat is None:
            return Node(value=get_most_common_el(y))
            
        if self.max_depth is not None and self.max_depth == depth:
            return Node(value=get_most_common_el(y))
            
        left_mask = X[best_feat] <= best_threshold
        right_mask = X[best_feat] > best_threshold

        left_child = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right_child = self._build_tree(X[right_mask], y[right_mask], depth + 1)

        return Node(left=left_child, right=right_child, feature=best_feat, threshold=best_threshold)

feats = df[['Fare', 'Sex']]
feats['Sex'] = feats['Sex'].apply(lambda x: 0 if x == 'male' else 1)
y = df["Survived"]
clf = MyDecisionTreeClassifier()
clf.fit(feats, y)

In [100]:
# with max_depth and different criterion
from IPython.display import HTML

class MyDecisionTreeClassifier:
    def __init__(self, criterion='gini', max_depth=None):
        if max_depth is not None and not isinstance(max_depth, int):
            raise TypeError('max_depth must be int')
        if max_depth is not None and max_depth < 1:
            raise ValueError('max_depth must be more or equal to one (1)')
            
        self.criterion = criterion
        self.root = None
        self.max_depth = max_depth

    def _split(self, X, y):
        return best_split(X, y)
        
    def fit(self, X, y):
        self.root = self._build_tree(X, y, 0)
        return self

    def _build_tree(self, X, y, depth):
        best_info_gain, best_threshold, best_feat = self._split(X, y)

        if best_feat is None:
            return Node(value=get_most_common_el(y))
            
        if self.max_depth is not None and self.max_depth == depth:
            return Node(value=get_most_common_el(y))
            
        left_mask = X[best_feat] <= best_threshold
        right_mask = X[best_feat] > best_threshold

        left_child = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right_child = self._build_tree(X[right_mask], y[right_mask], depth + 1)

        return Node(left=left_child, right=right_child, feature=best_feat, threshold=best_threshold)

    def _generate_node_html(self, node):
        """Рекурсивно строит HTML-код для каждого узла дерева."""
        if node is None:
            return ""

        # Если это ЛИСТ (предсказание)
        if node.value is not None:
            # Стилизуем лист в оранжевый/бежевый цвет, как в sklearn
            return f"""
            <div class="tree-node tree-leaf">
                <div class="node-title">🍃 Лист</div>
                <div class="node-details">Класс: <b>{node.value}</b></div>
            </div>
            """

        # Если это УЗЕЛ (условие)
        # Рекурсивно получаем HTML для левой и правой веток
        left_html = self._generate_node_html(node.left)
        right_html = self._generate_node_html(node.right)

        # Стилизуем узел со стрелочками к потомкам
        return f"""
        <div class="tree-branch">
            <div class="tree-node tree-split">
                <div class="node-title">🪵 Узел сплита</div>
                <div class="node-details"><b>{node.feature}</b> &le; {node.threshold:.2f}</div>
            </div>
            <div class="tree-children">
                <div class="tree-child-container">
                    <span class="branch-label label-left">Да (&le;)</span>
                    {left_html}
                </div>
                <div class="tree-child-container">
                    <span class="branch-label label-right">Нет (&gt;)</span>
                    {right_html}
                </div>
            </div>
        </div>
        """

    def plot_tree(self):
        """Отрисовывает компактную и миниатюрную структуру дерева."""
        css_styles = """
        <style>
            .tree-wrapper {
                display: flex;
                flex-direction: column;
                align-items: center;
                font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
                background-color: #fafafa;
                padding: 10px;
                overflow-x: auto;
                border-radius: 6px;
                border: 1px solid #eaeaea;
            }
            .tree-branch {
                display: flex;
                flex-direction: column;
                align-items: center;
            }
            .tree-node {
                padding: 4px 8px;
                border-radius: 4px;
                min-width: 90px;
                text-align: center;
                box-shadow: 0 1px 3px rgba(0,0,0,0.05);
                border: 1px solid;
                margin-bottom: 8px;
                background: #fff;
            }
            .tree-split {
                border-color: #2b7bba;
                background-color: #edf7ff;
            }
            .tree-leaf {
                border-color: #cca43b;
                background-color: #fff9eb;
            }
            .node-title {
                font-size: 9px;
                text-transform: uppercase;
                color: #888;
                margin-bottom: 1px;
            }
            .node-details {
                font-size: 11px;
                color: #222;
            }
            .tree-children {
                display: flex;
                justify-content: center;
                align-items: flex-start;
                gap: 15px; /* Уменьшили расстояние между ветками */
            }
            .tree-child-container {
                display: flex;
                flex-direction: column;
                align-items: center;
                padding-top: 4px;
            }
            .branch-label {
                font-size: 9px;
                font-weight: bold;
                padding: 1px 4px;
                background: #f0f0f0;
                border-radius: 3px;
                margin-bottom: 4px;
                color: #666;
            }
        </style>
        """
        tree_html = self._generate_node_html(self.root)
        full_html = f'{css_styles}<div class="tree-wrapper">{tree_html}</div>'
        return HTML(full_html)

feats = df[['Fare', 'Sex']]
feats['Sex'] = feats['Sex'].apply(lambda x: 0 if x == 'male' else 1)
y = df["Survived"]
clf = MyDecisionTreeClassifier(max_depth=4)
clf.fit(feats, y).plot_tree()

In [126]:
# with max_depth and different criterion
class MyDecisionTreeClassifier:
    def __init__(self, criterion='gini', max_depth=None):
        if max_depth is not None and not isinstance(max_depth, int):
            raise TypeError('max_depth must be int')
        if max_depth is not None and max_depth < 1:
            raise ValueError('max_depth must be more or equal to one (1)')
            
        self.criterion = criterion
        self.root = None
        self.max_depth = max_depth

    def fit(self, X, y):
        self.root = self._build_tree(X, y, 0)
        return self
        
    def _split(self, X, y):
        return best_split(X, y)

    def _build_tree(self, X, y, depth):
        best_info_gain, best_threshold, best_feat = self._split(X, y)

        if best_feat is None:
            return Node(value=get_most_common_el(y))
            
        if self.max_depth is not None and self.max_depth == depth:
            return Node(value=get_most_common_el(y))
            
        left_mask = X[best_feat] <= best_threshold
        right_mask = X[best_feat] > best_threshold

        left_child = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right_child = self._build_tree(X[right_mask], y[right_mask], depth + 1)

        return Node(left=left_child, right=right_child, feature=best_feat, threshold=best_threshold)

    def _get_value_by_series(self, sample, node):
        if node.value is not None:
            return node.value
    
        sample_value_by_feature = sample[node.feature]
        
        if sample_value_by_feature <= node.threshold:
            return self._get_value_by_series(sample, node.left)
    
        return self._get_value_by_series(sample, node.right)

    def predict(self, X):
        return np.array([self._get_value_by_series(sample, self.root) for sample in X.iloc])

feats = df[['Fare', 'Sex']]
feats['Sex'] = feats['Sex'].apply(lambda x: 0 if x == 'male' else 1)
y = df["Survived"]
clf = MyDecisionTreeClassifier()
clf.fit(feats, y)
clf.predict(feats)

array([0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0,
       1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1,
       0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0,
       1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0,
       0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0,
       0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0,
       1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0,
       0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0,
       0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0,
       1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0,
       1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1,
       1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1,
       1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0,

In [110]:
feats = df[['Fare', 'Sex']]
feats.iloc[0]

Fare    7.25
Sex     male
Name: 0, dtype: object

In [112]:
def get_value_by_series(sample, node):
    if node['value'] is not None:
        return node['value']

    sample_value_by_feature = sample[node['feature']]]
    
    if sample_value_by_feature <= node['threshold']:
        return get_value_by_series(sample, node["left"])

    return get_value_by_series(sample, node["right"])

return np.array([get_value_by_series(sample, self.root) for sample in feats.iloc])
    

<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.S